# OpenCode Continuum — Kaggle

Runtime **efímero** + workstation **persistente** (Kaggle Dataset).

### Secrets (Add-ons / Settings → Secrets, adjuntados a este cuaderno)

- `OPENCODE_CLOUD_DATASET` = `usuario/dataset-privado` (REQUERIDO)
- `OPENCODE_SERVER_PASSWORD` (recomendado, basic auth de OpenCode)
- `NVIDIA_API_KEY` (opcional — sin él arranca igual, sin modelos NVIDIA)
- `CLOUDFLARE_TUNNEL_TOKEN` (opcional — túnel nombrado permanente)

### Flujo

1. Instala el paquete desde GitHub (`opencode-continuum@main`)
2. Bootstrap restaura el Dataset y arranca OpenCode Web (puerto 4096)
3. Cloudflare Tunnel (o proxy Kaggle como fallback)
4. Imprime **un solo link**: tu última sesión activa (pin persistente)
5. La última celda (keep-alive) mantiene la sesión viva; si OpenCode
   no se recupera, termina para que el guardián externo relance el kernel

Bookmark permanente opcional: túnel nombrado + regla `/go` (ver README).
La URL quick-tunnel es temporal: no la guardes.


In [ ]:
!pip install -q kagglehub
!pip install -q git+https://github.com/enrrutador/opencode-continuum.git@main


In [ ]:
from opencode_kaggle.bootstrap import bootstrap
import os

# Config no-sensible embebida: el nombre del Dataset privado.
# Un secret `OPENCODE_CLOUD_DATASET` adjuntado pisa este default.
os.environ.setdefault("OPENCODE_CLOUD_DATASET", "marceloate/opencode-cloud-state")

info = bootstrap()
print("status:", info.get("status"))
print("recovery:", info.get("recovery"))
print("redirector_port:", info.get("redirector_port"))
wa = info.get("web_access") or {}
print("provider:", wa.get("provider"))
print("opencode_listening:", wa.get("opencode_listening"))
print("access status:", wa.get("status"))
if not info.get("ok"):
    print("message:", info.get("message"))


In [ ]:
# Un solo link: última sesión activa + pin persistente en el Dataset.
# El historial completo de sesiones queda visible dentro de la web.
import base64
import json
import os
import re
import urllib.parse
import urllib.request
from pathlib import Path

from opencode_cloud.entry import load_pin, pick_session, save_pin

port = info.get("opencode_port", 4096)
wa = info.get("web_access") or {}
base_url = (wa.get("url") or "").rstrip("/")
if not base_url:
    lp = Path("/kaggle/working/opencode_cloud/logs/cloudflare-tunnel.log")
    if lp.exists():
        m = re.search(r"https://[a-zA-Z0-9.-]+\.(?:trycloudflare|cfargotunnel)\.com[^\s]*", lp.read_text())
        if m:
            base_url = m.group(0).rstrip("/")
if not base_url:
    raise RuntimeError("Sin URL de túnel")

workspace = info.get("workspace") or "/kaggle/working/opencode_cloud/workspace"
ws_b64 = base64.urlsafe_b64encode(str(workspace).encode()).decode().rstrip("=")

def open_api(method, path, data=None):
    req = urllib.request.Request(
        f"http://127.0.0.1:{port}{path}", data=data, method=method
    )
    req.add_header("Content-Type", "application/json")
    pw = os.environ.get("OPENCODE_SERVER_PASSWORD", "")
    un = os.environ.get("OPENCODE_SERVER_USERNAME", "opencode")
    if pw:
        tok = base64.b64encode(f"{un}:{pw}".encode()).decode()
        req.add_header("Authorization", f"Basic {tok}")
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode() or "null")

pin_path = Path("/kaggle/working/opencode_cloud/metadata/session.json")
query = "?directory=" + urllib.parse.quote(str(workspace))
sessions = []
try:
    got = open_api("GET", "/session" + query)
    sessions = got if isinstance(got, list) else []
except Exception as e:
    print("[link] GET /session:", type(e).__name__)

if not sessions:
    try:
        seed = open_api("POST", "/session" + query, b"{}")
        if seed and seed.get("id"):
            sessions = [seed]
            print("[link] sesión seed:", seed.get("id"))
    except Exception as e:
        print("[link] POST /session:", type(e).__name__)

chosen = pick_session(sessions, load_pin(pin_path))
if chosen:
    OPEN_URL = f"{base_url}/{ws_b64}/session/{chosen['id']}"
    OPEN_TITLE = chosen.get("title") or chosen["id"]
    save_pin(pin_path, chosen["id"], OPEN_TITLE)
else:
    OPEN_URL = f"{base_url}/{ws_b64}/session"
    OPEN_TITLE = "(nueva conversación)"

out = Path("/kaggle/working/opencode_cloud/OPEN_THIS_URL.txt")
out.write_text(
    OPEN_URL
    + "\n\nsesion: " + OPEN_TITLE
    + "\nrecovery: " + str(info.get("recovery", ""))
    + "\nsesiones: " + str(len(sessions)) + "\n",
    encoding="utf-8",
)

print("######## TU LINK (una sola línea) ########")
print(OPEN_URL)
print("##########################################")
print("sesión:", OPEN_TITLE, "| historial:", len(sessions))
if wa.get("provider") == "cloudflare_named_tunnel":
    print("Bookmark permanente vía regla /go del túnel (ver README del repo).")


In [ ]:
# Keep-alive: mantiene esta celda ejecutando para que Kaggle no mate
# la sesión por inactividad. Si OpenCode deja de responder y el watchdog
# interno no lo revive en ~5 min, la celda termina: el guardián externo
# (GitHub Actions / scripts/guardian) relanza el kernel y todo se restaura.
import socket
import time
from datetime import datetime, timezone

from opencode_cloud.ports import get_opencode_port

PORT = get_opencode_port()
TICK_S = 60
GRACE_TICKS = 5
BEAT_TICKS = 10

misses = 0
ticks = 0
while True:
    s = socket.socket()
    s.settimeout(5)
    try:
        s.connect(("127.0.0.1", PORT))
        alive = True
    except OSError:
        alive = False
    finally:
        s.close()
    ticks += 1
    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    if alive:
        misses = 0
        if ticks % BEAT_TICKS == 1:
            print(f"[keep-alive] {now} OpenCode responde en {PORT}", flush=True)
    else:
        misses += 1
        print(f"[keep-alive] {now} puerto {PORT} sin respuesta ({misses}/{GRACE_TICKS})", flush=True)
        if misses > GRACE_TICKS:
            print("[keep-alive] sin recuperación: termino para que el guardián relance el kernel.", flush=True)
            break
    time.sleep(TICK_S)
